# 01 — Auditoria do TechQA

Visualização do manifesto produzido por `src/rag_confidence/data/audit.py`. O notebook não altera arquivos brutos nem implementa regras de preparação.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
MANIFEST = ROOT / 'data' / 'manifests' / 'techqa_audit.json'
audit = json.loads(MANIFEST.read_text(encoding='utf-8'))
audit['source']

## Composição dos splits

In [ ]:
split_rows = []
for name, values in audit['splits'].items():
    split_rows.append({
        'split': name,
        'total': values['count'],
        'respondível': values['answerability'].get('Y', 0),
        'não respondível': values['answerability'].get('N', 0),
        'spans divergentes': values['span_mismatches'],
    })
split_df = pd.DataFrame(split_rows).set_index('split')
split_df

In [ ]:
ax = split_df[['respondível', 'não respondível']].plot.bar(
    stacked=True, figsize=(8, 4), color=['#2f6f5e', '#b85c38']
)
ax.set(title='Composição dos conjuntos auditados', xlabel='', ylabel='Perguntas')
ax.legend(frameon=False)
plt.tight_layout();

## Sobreposição entre conjuntos

In [ ]:
overlap_df = pd.DataFrame(audit['exact_question_overlap'])[[
    'pair', 'exact_question_text_matches', 'answerability_label_conflicts'
]]
overlap_df

Os pares próximos são candidatos de revisão, não exclusões automáticas.

In [ ]:
pd.DataFrame([
    {
        'pair': item['pair'],
        'limiar': item['threshold'],
        'pares': item['count'],
        'perguntas à esquerda': item['unique_left_queries'],
        'perguntas à direita': item['unique_right_queries'],
    }
    for item in audit['near_duplicate_question_overlap']
])